# Preparação dos Dados — Campeonato Brasileiro

Este notebook carrega, limpa e divide os dados históricos do Brasileirão (2003–2025) em dois arquivos parquet para consumo pelos notebooks de modelagem (Fases 2–4).

## 1. Configuração e importações

In [1]:
import os

# Configuração para renderização headless (sem display gráfico)
# Deve ser feita ANTES de qualquer importação de matplotlib/seaborn
os.environ['MPLBACKEND'] = 'agg'
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib-config'

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow

print(f"pandas {pd.__version__}")
print(f"pyarrow {pyarrow.__version__}")
print(f"seaborn {sns.__version__}")

pandas 2.3.3
pyarrow 24.0.0
seaborn 0.13.2


## 2. Carregamento do CSV

In [2]:
df = pd.read_csv('dados/archive/campeonato-brasileiro-full.csv')

assert len(df) == 9165, f"Esperado 9165 linhas, obtido {len(df)}"
assert list(df.columns) == [
    'ID', 'rodata', 'data', 'hora', 'mandante', 'visitante',
    'formacao_mandante', 'formacao_visitante', 'tecnico_mandante',
    'tecnico_visitante', 'vencedor', 'arena', 'mandante_Placar',
    'visitante_Placar', 'mandante_Estado', 'visitante_Estado', 'arrecadacao'
], f"Colunas inesperadas: {list(df.columns)}"

print(f"shape: {df.shape}")
print(df.dtypes)

shape: (9165, 17)
ID                      int64
rodata                  int64
data                   object
hora                   object
mandante               object
visitante              object
formacao_mandante      object
formacao_visitante     object
tecnico_mandante       object
tecnico_visitante      object
vencedor               object
arena                  object
mandante_Placar         int64
visitante_Placar        int64
mandante_Estado        object
visitante_Estado       object
arrecadacao           float64
dtype: object


## 3. Parse da coluna de data e verificação de ordem cronológica

In [3]:
df['date'] = pd.to_datetime(df['data'], format='%d/%m/%Y')

assert df['date'].is_monotonic_increasing, "CSV não está ordenado por data — ordenação necessária"
assert df['date'].isnull().sum() == 0, "Valores nulos encontrados na coluna 'date'"

print(f"Data mínima: {df['date'].min()}")
print(f"Data máxima: {df['date'].max()}")

Data mínima: 2003-03-29 00:00:00
Data máxima: 2025-12-07 00:00:00


## 4. Derivação da coluna alvo `result`

In [4]:
def derive_result(row):
    if row['vencedor'] == '-':
        return 'Draw'
    elif row['vencedor'] == row['mandante']:
        return 'HomeWin'
    else:
        return 'AwayWin'

df['result'] = df.apply(derive_result, axis=1)

assert df['result'].isnull().sum() == 0, "Valores nulos em 'result'"
assert set(df['result'].unique()) == {'HomeWin', 'Draw', 'AwayWin'}, f"Valores inesperados: {set(df['result'].unique())}"

print(df['result'].value_counts())

result
HomeWin    4550
Draw       2421
AwayWin    2194
Name: count, dtype: int64


## 5. Renomeação de colunas, remoção das não utilizadas e adição de `season`

In [5]:
df = df.rename(columns={
    'mandante':        'home_team',
    'visitante':       'away_team',
    'mandante_Placar': 'home_score',
    'visitante_Placar':'away_score',
    'rodata':          'round',
    'mandante_Estado': 'home_state',
    'visitante_Estado':'away_state',
})

df['season'] = df['date'].dt.year

# Selecionar apenas as colunas necessárias — colunas com muitos nulos
# (formacao_*, tecnico_*, arena, arrecadacao) são excluídas por seleção de colunas,
# NÃO por df.dropna(), que removeria ~96% das linhas
keep_cols = ['ID', 'date', 'season', 'round', 'home_team', 'away_team',
             'home_score', 'away_score', 'home_state', 'away_state', 'result']
df = df[keep_cols]

assert df.isnull().sum().sum() == 0, f"Valores nulos encontrados após seleção: {df.isnull().sum()}"
assert list(df.columns) == keep_cols, f"Colunas inesperadas: {list(df.columns)}"

print(f"shape: {df.shape}")
print(df.head(2))

shape: (9165, 11)
   ID       date  season  round     home_team away_team  home_score  \
0   1 2003-03-29    2003      1       Guarani     Vasco           4   
1   2 2003-03-29    2003      1  Athletico-PR    Gremio           2   

   away_score home_state away_state   result  
0           2         SP         RJ  HomeWin  
1           0         PR         RS  HomeWin  


## 6. Normalização dos nomes dos times e verificação dos 10 clubes principais

In [6]:
# Nomes no CSV já são consistentes (46 nomes únicos, 0 conflitos).
# O dicionário é uma guarda defensiva — popule-o quando variantes aparecerem
# (ex.: 'Atletico MG': 'Atletico-MG' — não presente atualmente)
canonical_names = {}

for col in ['home_team', 'away_team']:
    df[col] = df[col].replace(canonical_names)

team_counts = pd.concat([df['home_team'], df['away_team']]).value_counts()

top_clubs = [
    'Flamengo', 'Corinthians', 'Sao Paulo', 'Fluminense', 'Santos',
    'Internacional', 'Atletico-MG', 'Athletico-PR', 'Gremio', 'Palmeiras'
]

for club in top_clubs:
    assert team_counts.get(club, 0) >= 380, f"{club}: {team_counts.get(club, 0)} < 380"

print(team_counts.head(15))

Flamengo         894
Fluminense       894
Sao Paulo        894
Santos           856
Internacional    856
Corinthians      856
Atletico-MG      855
Athletico-PR     818
Gremio           814
Palmeiras        810
Cruzeiro         780
Botafogo-RJ      772
Vasco            704
Goias            590
Coritiba         590
Name: count, dtype: int64


## 7. Divisão temporal treino/teste

In [7]:
train_df = df[df['date'].dt.year <= 2022].copy()
test_df  = df[df['date'].dt.year >= 2023].copy()

assert len(train_df) == 8025, f"Esperado 8025 linhas de treino, obtido {len(train_df)}"
assert len(test_df)  == 1140, f"Esperado 1140 linhas de teste, obtido {len(test_df)}"
assert len(train_df) + len(test_df) == 9165, "Soma treino+teste difere do total"
assert len(set(train_df.index) & set(test_df.index)) == 0, "Sobreposição de índices entre treino e teste!"
assert train_df['date'].max() < test_df['date'].min(), "Sobreposição temporal entre treino e teste!"

print(f"Treino: {len(train_df)} linhas ({train_df['date'].dt.year.min()}–{train_df['date'].dt.year.max()})")
print(f"Teste:  {len(test_df)} linhas ({test_df['date'].dt.year.min()}–{test_df['date'].dt.year.max()})")

Treino: 8025 linhas (2003–2022)
Teste:  1140 linhas (2023–2025)


## 8. Salvamento dos arquivos parquet

In [8]:
os.makedirs('dados', exist_ok=True)

train_df.to_parquet('dados/matches_train.parquet', engine='pyarrow', index=False)
test_df.to_parquet('dados/matches_test.parquet',   engine='pyarrow', index=False)

assert os.path.exists('dados/matches_train.parquet'), "matches_train.parquet não encontrado!"
assert os.path.exists('dados/matches_test.parquet'),  "matches_test.parquet não encontrado!"

# Leitura de verificação (round-trip)
reload_train = pd.read_parquet('dados/matches_train.parquet')
expected_cols = ['ID', 'date', 'season', 'round', 'home_team', 'away_team',
                 'home_score', 'away_score', 'home_state', 'away_state', 'result']
assert len(reload_train) == 8025, f"Round-trip: esperado 8025, obtido {len(reload_train)}"
assert list(reload_train.columns) == expected_cols, f"Colunas inesperadas no parquet: {list(reload_train.columns)}"

print(f"Salvo: treino={len(train_df)} linhas, teste={len(test_df)} linhas")

Salvo: treino=8025 linhas, teste=1140 linhas
